# 02 — Dimensões Vetoriais: 384 vs 768 vs 1024

## O que é uma "dimensão"?

Um embedding é basicamente uma lista de números. O número de elementos dessa lista é o que chamamos de **dimensão**.

Por exemplo, o modelo  gera vetores com **384 números**:



Cada um desses 384 números é uma **dimensão**. Você pode pensar assim:

> **Coordenadas geográficas** usam 2 dimensões (latitude + longitude) para localizar qualquer ponto na Terra.
> **Embeddings** usam centenas de dimensões para localizar qualquer texto num "espaço de significados".

Você não consegue interpretar o que significa o número na posição 42 ou na posição 137 — o modelo aprendeu essas representações automaticamente durante o treinamento. O que importa é que, coletivamente, esses números capturam o **significado** do texto de forma que textos parecidos ficam próximos nesse espaço.

---

## Por que diferentes modelos têm dimensões diferentes?

| Dimensões | Intuição |
|-----------|----------|
| **Poucas (128–384)** | Menos "slots" pra guardar nuances. Mais rápido, menor uso de memória. Boa para casos simples. |
| **Médias (768)** | Equilíbrio entre qualidade e custo. Padrão da maioria dos modelos BERT. |
| **Muitas (1024–3072)** | Mais "slots" para capturar nuances sutis. Mais lento, mais memória. Melhor qualidade em tarefas difíceis. |

Neste notebook você vai ver **na prática** o que isso significa em termos de memória, velocidade e qualidade.

In [ ]:
import numpy as np
import time
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sentence_transformers import SentenceTransformer

# Modelos de diferentes dimensões
MODELS = {
    "all-MiniLM-L6-v2": {"dims": 384, "size_mb": 22, "params": "22M"},
    "all-mpnet-base-v2": {"dims": 768, "size_mb": 420, "params": "110M"},
    # Descomente se tiver memória suficiente (~1.3GB):
    # "all-roberta-large-v1": {"dims": 1024, "size_mb": 1300, "params": "355M"},
}

print("Modelos que vamos comparar:")
for name, info in MODELS.items():
    print(f"  {name}: {info['dims']}d | ~{info['size_mb']}MB | {info['params']} params")

## 2.1 Uso de Memória por Dimensão

Antes de tudo, um cálculo simples mas importante.

Cada número no vetor é armazenado como  — um número de ponto flutuante de 32 bits, ou seja, **4 bytes**.

Então o custo de memória de um único embedding é:



Parece pouco, mas imagina isso pra milhões de documentos:



Isso é o custo **só do índice vetorial**. E esse índice precisa caber em RAM para buscas rápidas — disco é lento demais.

A tabela abaixo mostra o custo real por escala:

In [ ]:
def calcular_memoria(n_vetores, dimensoes, dtype_bytes=4):
    """Calcula uso de memória em GB."""
    return (n_vetores * dimensoes * dtype_bytes) / (1024**3)

scenarios = [
    ("1K docs", 1_000),
    ("10K docs", 10_000),
    ("100K docs", 100_000),
    ("1M docs", 1_000_000),
    ("10M docs", 10_000_000),
]

dimensoes_list = [128, 384, 768, 1024, 1536, 3072]

rows = []
for label, n in scenarios:
    row = {"Escala": label}
    for d in dimensoes_list:
        gb = calcular_memoria(n, d)
        row[f"{d}d"] = f"{gb*1024:.1f}MB" if gb < 1 else f"{gb:.2f}GB"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Escala")
print("Uso de memória (float32) por escala e dimensão:\n")
print(df.to_string())

print("\n💡 Observações:")
print("  1M docs a 768d (MPNet) = 3.07GB — cabe num servidor moderno")
print("  1M docs a 3072d (OpenAI large) = 12.3GB — requer hardware dedicado")
print("  10M docs a 1536d (GPT ada) = 61.4GB — precisa de quantização!")

In [ ]:
# Visualização: crescimento de memória por dimensão
n_vetores = 1_000_000  # 1 milhão de documentos
dims_range = range(64, 3073, 64)
memoria_gb = [calcular_memoria(n_vetores, d) for d in dims_range]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dims_range, memoria_gb, 'b-', linewidth=2)
ax.fill_between(dims_range, memoria_gb, alpha=0.1)

# Marcar modelos conhecidos
marcadores = {384: "MiniLM\n384d", 768: "MPNet\n768d", 
              1024: "RoBERTa\n1024d", 1536: "OpenAI\nada-002", 3072: "OpenAI\nlarge"}
for dim, label in marcadores.items():
    mem = calcular_memoria(n_vetores, dim)
    ax.axvline(x=dim, color='red', linestyle='--', alpha=0.5)
    ax.annotate(f"{label}\n{mem:.1f}GB", (dim, mem),
                textcoords="offset points", xytext=(5, 10), fontsize=9, color='red')

ax.set_xlabel("Dimensões", fontsize=12)
ax.set_ylabel("Memória (GB) — 1M vetores, float32", fontsize=12)
ax.set_title("Uso de Memória vs Dimensão (float32, 1M documentos)", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.2 Benchmark: Velocidade de Embedding

Mais dimensões = mais parâmetros no modelo = mais computação pra gerar cada vetor.

Isso importa em dois cenários:

- **Indexação**: quando você precisa gerar embeddings para toda a sua base de documentos (pode ser milhões)
- **Busca em tempo real**: quando o usuário digita uma query e você precisa embeddar em < 100ms

Vamos medir quantos textos por segundo cada modelo consegue processar.

> **Nota:** esses números variam muito dependendo do hardware (CPU vs GPU) e tamanho do batch. O importante aqui é ver a **diferença relativa** entre os modelos.

In [ ]:
# Gerar textos de teste
import random

templates = [
    "A inteligência artificial está transformando {setor}.",
    "O modelo de linguagem {nome} foi treinado com {n} bilhões de tokens.",
    "Embeddings de {d} dimensões são usados em {app}.",
    "A empresa {empresa} anunciou novos investimentos em {tech}.",
]

fills = {
    "setor": ["saúde", "educação", "finanças", "varejo"],
    "nome": ["GPT-4", "Claude", "Gemini", "Llama"],
    "n": ["1", "10", "100", "1000"],
    "d": ["384", "768", "1024", "1536"],
    "app": ["busca semântica", "RAG", "recomendação"],
    "empresa": ["Google", "Microsoft", "Anthropic", "Meta"],
    "tech": ["LLMs", "RAG", "vector databases"],
}

random.seed(42)
test_texts = []
for _ in range(500):
    template = random.choice(templates)
    for key, options in fills.items():
        template = template.replace("{" + key + "}", random.choice(options))
    test_texts.append(template)

print(f"Gerados {len(test_texts)} textos de teste")
print(f"Exemplos: {test_texts[:2]}")

In [ ]:
benchmark_results = {}

for model_name, info in MODELS.items():
    print(f"\nCarregando {model_name}...")
    m = SentenceTransformer(model_name)
    
    # Warm-up
    _ = m.encode(["warm up"])
    
    # Benchmark
    start = time.perf_counter()
    embs = m.encode(test_texts, batch_size=32, show_progress_bar=False)
    elapsed = time.perf_counter() - start
    
    texts_per_sec = len(test_texts) / elapsed
    memory_per_1m = calcular_memoria(1_000_000, info['dims'])
    
    benchmark_results[model_name] = {
        "dims": info['dims'],
        "texts_per_sec": texts_per_sec,
        "total_time_s": elapsed,
        "memory_1m_gb": memory_per_1m,
    }
    
    print(f"  ✅ {info['dims']}d | {texts_per_sec:.0f} textos/s | {memory_per_1m:.2f}GB/1M vetores")
    del m  # liberar memória

In [ ]:
# Tabela resumo
df_bench = pd.DataFrame(benchmark_results).T
df_bench["texts_per_sec"] = df_bench["texts_per_sec"].astype(float).round(0)
df_bench["memory_1m_gb"] = df_bench["memory_1m_gb"].astype(float).round(2)

# Normalizar em relação ao MiniLM
baseline_speed = df_bench.loc["all-MiniLM-L6-v2", "texts_per_sec"]
df_bench["speed_relative"] = (df_bench["texts_per_sec"] / baseline_speed).round(2)

print("\n📊 Benchmark de Velocidade:\n")
print(df_bench[["dims", "texts_per_sec", "speed_relative", "memory_1m_gb"]].to_string())
print("\n(speed_relative: 1.0 = mesmo que MiniLM, 0.5 = metade da velocidade)")

## 2.3 Qualidade Semântica: Mais Dimensões = Melhor?

Memória e velocidade são fáceis de medir. Mas o que realmente importa é: **o modelo com mais dimensões encontra resultados mais relevantes?**

A intuição é que mais dimensões dão ao modelo mais "espaço" para codificar nuances:

- Com 384 dimensões, o modelo talvez não consiga separar bem "backpropagation" de "gradient descent" — ambos são conceitos de ML
- Com 768 dimensões, há mais espaço para capturar que um é um conceito teórico e o outro é um algoritmo

Na prática, a diferença **existe mas é menor do que você esperaria** para a maioria dos casos de uso.

Vamos testar com consultas reais e ver qual modelo encontra o documento mais relevante:

In [ ]:
# Conjunto de testes para avaliar qualidade semântica
test_queries = [
    "Como treinar um modelo de machine learning?",
    "Quais são os melhores restaurantes em São Paulo?",
    "O que é backpropagation?",
]

candidate_docs = [
    "O treinamento de redes neurais envolve otimização por gradiente descendente.",  # relevante para query 0 e 2
    "São Paulo tem uma culinária diversificada, com opções japonesas e italianas.",   # relevante para query 1
    "O algoritmo backpropagation calcula gradientes através da regra da cadeia.",    # relevante para query 2
    "A taxa de aprendizado é um hiperparâmetro crucial no treinamento.",             # relevante para query 0
    "O Japão tem culinária tradicional baseada em frutos do mar.",                  # levemente relevante para query 1
    "Python é a linguagem mais usada para ciência de dados.",                       # não muito relevante
]

print("Avaliando qualidade de retrieval por modelo...\n")

for model_name, info in MODELS.items():
    m = SentenceTransformer(model_name)
    
    query_embs = m.encode(test_queries, normalize_embeddings=True)
    doc_embs = m.encode(candidate_docs, normalize_embeddings=True)
    
    print(f"\n{'='*60}")
    print(f"Modelo: {model_name} ({info['dims']}d)")
    print(f"{'='*60}")
    
    for q_idx, query in enumerate(test_queries):
        scores = query_embs[q_idx] @ doc_embs.T
        top_idx = np.argsort(scores)[::-1][0]
        print(f"\nQuery: '{query}'")
        print(f"  Top-1: (score={scores[top_idx]:.3f}) {candidate_docs[top_idx][:70]}...")
    
    del m

## 2.4 A Maldição da Dimensionalidade

Existe um fenômeno contraintuitivo chamado **"maldição da dimensionalidade"**:

> Quanto mais dimensões um espaço tem, mais difícil fica distinguir pontos próximos de pontos distantes.

**Por quê?** Imagine você no centro de um círculo 2D. A maioria dos pontos aleatórios fica espalhada pelo círculo. Agora imagine uma esfera 3D — os pontos se concentram na superfície, não no centro. Em 1000 dimensões, praticamente todos os pontos aleatórios ficam numa camada fina perto da superfície — e a distância entre todos eles começa a ser quase igual.

**Consequência para busca vetorial**: se todas as distâncias são parecidas, fica difícil dizer o que é "próximo" e o que é "distante". O índice vetorial perde poder discriminativo.

**Por que isso não destrói os embeddings de texto?**

1. **Os dados não são aleatórios** — textos em português vivem numa região muito específica do espaço, não preenchem todo ele
2. **Sub-variedade de baixa dimensão** — apesar de 768 dimensões existirem, a estrutura real dos dados pode ser muito mais simples (como uma folha dobrada num espaço 3D)
3. **Normalização** — usar vetores na esfera unitária (‖v‖=1) concentra os pontos numa superfície, o que ajuda

O gráfico abaixo mostra esse efeito em vetores **puramente aleatórios** (sem estrutura semântica):

In [ ]:
# Demonstração: concentração de normas em alta dimensão
np.random.seed(42)

n_samples = 10000
dimensoes_demo = [2, 10, 100, 384, 768, 1536]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: distribuição de distâncias cosine entre pares aleatórios
for d in dimensoes_demo:
    # Vetores gaussianos aleatórios (sem estrutura semântica)
    vecs = np.random.randn(n_samples, d).astype(np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    vecs_norm = vecs / norms
    
    # Amostra de 1000 pares aleatórios
    idx = np.random.choice(n_samples, size=1000, replace=False)
    cos_sims = vecs_norm[idx[:500]] @ vecs_norm[idx[500:]].T
    diag_sims = np.diag(cos_sims)
    
    axes[0].hist(diag_sims, bins=50, alpha=0.5, density=True, label=f"{d}d")

axes[0].set_title("Distribuição de Similaridade Cosine\n(vetores aleatórios sem estrutura)")
axes[0].set_xlabel("Similaridade Cosine")
axes[0].set_ylabel("Densidade")
axes[0].legend(fontsize=8)
axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.5)

# Plot 2: variância das distâncias por dimensão
all_dims = range(2, 1001, 20)
variances = []
for d in all_dims:
    vecs = np.random.randn(500, d).astype(np.float32)
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    sims = vecs @ vecs.T
    # Pegar apenas triângulo superior (exceto diagonal)
    mask = np.triu(np.ones_like(sims, dtype=bool), k=1)
    variances.append(np.var(sims[mask]))

axes[1].semilogy(list(all_dims), variances, 'b-', linewidth=2)
axes[1].set_title("Variância das Distâncias vs Dimensão\n(↓ variância = mais difícil distinguir pontos)")
axes[1].set_xlabel("Dimensões")
axes[1].set_ylabel("Variância das Similaridades (log scale)")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Maldição da Dimensionalidade (em dados puramente aleatórios)", 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Para embeddings REAIS, esse efeito é atenuado porque:")
print("   1. Os dados vivem numa sub-variedade de baixa dimensão")
print("   2. O modelo aprendeu a organizar o espaço semanticamente")

## Resumo: Quando usar cada dimensão?

| Dimensões | Modelo | Quando usar |
|-----------|--------|-------------|
| **384** | all-MiniLM-L6-v2 | Prototipagem, tempo real, edge devices |
| **768** | all-mpnet-base-v2 | Produção balanceada, maioria dos casos |
| **1024** | all-roberta-large-v1 | Tarefas exigentes de qualidade |
| **1536** | OpenAI ada-002 | API, sem infra local |
| **3072** | OpenAI text-3-large | State-of-the-art, custo alto |

## Regra prática

> Para a maioria dos sistemas RAG de produção:
> **768d (all-mpnet-base-v2) é o ponto ideal entre qualidade e custo.**

## Próximos passos

- [03 — Float Types](03_float_types.html): Comprimir de 3GB para 768MB sem perder qualidade